# Tissue Detection Failure Behind Validation Collapse

This notebook documents the root cause of the BRCA validation collapse: the old tissue-detection tiling padded edge thumbnails differently from the GrandQC reference script. On smaller thumbnails, that caused the tissue detector to return almost no class-1 normal tissue, so the artifact stage saw background instead of tissue.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
from PIL import Image
from modules.validation import align_prediction_to_reference, extract_tcga_slide_id

REF_DIR = REPO_ROOT / 'data' / 'reference_brca' / 'reference_masks'
BEFORE_DIR = REPO_ROOT / 'data' / 'reference_brca' / 'predicted_masks'
AFTER_DIR = REPO_ROOT / 'outputs' / 'validation' / 'regenerated_masks'
BEFORE_AFTER = REPO_ROOT / 'outputs' / 'validation' / 'tissue_detection_before_after.parquet'

## Before/After Evidence

The key signature is normal-tissue class collapse. The three failing slides had 0 or near-0 class-1 pixels before the fix; after the tissue tiling fix they regain millions of normal-tissue pixels and high pixel agreement.

In [ ]:
before_after = pd.read_parquet(BEFORE_AFTER)
before_after

## Validation Metrics After Fix

These are regenerated through the fixed direct-DICOM path. The artifact model and artifact scoring path were not changed.

In [ ]:
per_slide = pd.read_parquet(REPO_ROOT / 'outputs' / 'validation' / 'validation_per_slide.parquet')
per_class = pd.read_parquet(REPO_ROOT / 'outputs' / 'validation' / 'validation_per_class.parquet')
display(per_slide)
display(per_class.groupby(['class_id','class_name']).agg(dice_mean=('dice','mean'), dice_std=('dice','std'), ref_px=('reference_px','sum')).reset_index())

## Recompute The Before/After Table

This cell recomputes the table from reference masks, cached old predictions, and regenerated predictions.

In [ ]:
def load_mask(path):
    arr = np.asarray(Image.open(path))
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr.astype('uint8')

refs = {extract_tcga_slide_id(path): path for path in REF_DIR.glob('*.png')}
before = {extract_tcga_slide_id(path): path for path in BEFORE_DIR.glob('*.png')}
after = {extract_tcga_slide_id(path): path for path in AFTER_DIR.glob('*.png')}
rows = []
for slide_id in sorted(refs):
    ref = load_mask(refs[slide_id])
    old = align_prediction_to_reference(ref, load_mask(before[slide_id]))
    new = align_prediction_to_reference(ref, load_mask(after[slide_id]))
    valid_old = np.isin(ref, range(1, 8)) & np.isin(old, range(1, 8))
    valid_new = np.isin(ref, range(1, 8)) & np.isin(new, range(1, 8))
    rows.append({
        'slide_id': slide_id,
        'before_agreement': float((ref[valid_old] == old[valid_old]).mean()),
        'after_agreement': float((ref[valid_new] == new[valid_new]).mean()),
        'ref_normal_tissue_px': int((ref == 1).sum()),
        'before_normal_tissue_px': int((old == 1).sum()),
        'after_normal_tissue_px': int((new == 1).sum()),
    })
pd.DataFrame(rows)

## Root Cause

The previous wrapper padded partial tissue-detection tiles from the top-left, while GrandQC's reference script crops right/bottom edge tiles from `width - 512` and `height - 512`, always feeding full 512 x 512 real-image edge crops when possible. On smaller tissue thumbnails this difference was enough to make the tissue detector classify tissue as background. The fix mirrors the reference edge-crop behavior for tissue detection only; artifact model loading, artifact inference, and artifact scoring were not changed.